# Perturb-Seqr GraphQL API Examples

This notebook demonstrates how to query the Perturb-Seqr GraphQL API programmatically.

In [ ]:
import requests
import pandas as pd

URL = "http://localhost:3000/graphql"

## 1. Single Gene Set Enrichment

Query the Perturb-Seqr database for gene sets that significantly overlap with your query gene set.
The optional `library_names` parameter filters results to specific datasets.

In [ ]:
def enrich_single_set(
    geneset: list[str],
    first: int = 100,
    library_names: list[str] | None = None,
):
    """Single gene set enrichment against Perturb-Seqr."""
    variables: dict = {
        "filterTerm": "",
        "offset": 0,
        "first": first,
        "filterFda": False,
        "sortBy": "pvalue_up",
        "filterKo": False,
        "genes": geneset,
    }
    if library_names is not None:
        variables["libraryNames"] = library_names

    query = {
        "operationName": "EnrichmentQuery",
        "variables": variables,
        "query": """query EnrichmentQuery(
            $genes: [String]!
            $filterTerm: String = \"\"
            $offset: Int = 0
            $first: Int = 10
            $filterFda: Boolean = false
            $sortBy: String = \"\"
            $filterKo: Boolean = false
            $libraryNames: [String]
        ) {
            currentBackground {
                enrich(
                    genes: $genes
                    filterTerm: $filterTerm
                    offset: $offset
                    first: $first
                    filterFda: $filterFda
                    sortby: $sortBy
                    filterKo: $filterKo
                    libraryNames: $libraryNames
                ) {
                    nodes {
                        geneSetHash
                        pvalue
                        adjPvalue
                        oddsRatio
                        nOverlap
                        geneSets {
                            nodes {
                                term
                                id
                                nGeneIds
                                geneSetFdaCountsById {
                                    nodes { approved count }
                                }
                            }
                            totalCount
                        }
                    }
                    totalCount
                    geneSetCount
                    consensusCount
                    consensus {
                        drug
                        oddsRatio
                        pvalue
                        adjPvalue
                        approved
                        countSignificant
                        countInsignificant
                        countUpSignificant
                        pvalueUp adjPvalueUp oddsRatioUp
                        pvalueDown adjPvalueDown oddsRatioDown
                    }
                }
            }
        }""",
    }

    response = requests.post(URL, json=query)
    response.raise_for_status()
    res = response.json()

    enrich = res["data"]["currentBackground"]["enrich"]
    df_consensus = pd.DataFrame(enrich["consensus"]).rename(
        columns={"drug": "perturbation"}
    )
    df_enrichment = pd.json_normalize(
        enrich["nodes"],
        record_path=["geneSets", "nodes"],
        meta=["geneSetHash", "pvalue", "adjPvalue", "oddsRatio", "nOverlap"],
    )
    if df_enrichment.empty:
        return pd.DataFrame(), pd.DataFrame()

    df_enrichment["direction"] = df_enrichment["term"].map(
        lambda t: t.split(" ")[-1]
    )
    return df_enrichment, df_consensus, enrich["geneSetCount"], enrich["totalCount"]

In [ ]:
genes = ["STAT1", "STAT3", "JAK1", "JAK2", "IFNG", "IRF1", "SOCS1", "SOCS3"]

# Enrich against all datasets
df_enrichment, df_consensus, gene_set_count, total_count = enrich_single_set(genes)
print(f"Background gene sets: {gene_set_count}")
print(f"Significant results: {total_count}")
df_enrichment.head(10)

### Dataset Filtering via `libraryNames`

Restrict results to specific datasets. Each gene set is tested against its own dataset-specific background, so p-values remain the same regardless of which datasets are selected.

Available dataset names:
- **Drug perturbation:** LINCS L1000 CP, Tahoe-100M, NIBR DRUG-seq, Microarrays CMap, Ginkgo Bioworks, SciPlex, DeepCover MoA, CREEDS Chem, RummaGEO Chem
- **Gene perturbation:** LINCS L1000 XPR, Perturb Atlas, Perturb Atlas Mouse, Replogle et al., CREEDS Gene, RummaGEO Gene

In [ ]:
# Enrich against only two drug perturbation datasets
df_filtered, _, filtered_count, filtered_total = enrich_single_set(
    genes, library_names=["LINCS L1000 CP", "SciPlex"]
)
print(f"Filtered background gene sets: {filtered_count}")
print(f"Filtered significant results: {filtered_total}")
df_filtered.head(10)

### Consensus Results

In [ ]:
df_consensus.head(10)

## 2. Up & Down Gene Set Enrichment

Find signatures that mimic or reverse the submitted expression pattern.

In [ ]:
def enrich_up_down(
    genes_up: list[str],
    genes_down: list[str],
    first: int = 100,
):
    """Up and down gene set enrichment against Perturb-Seqr."""
    query = {
        "operationName": "PairEnrichmentQuery",
        "variables": {
            "filterTerm": "",
            "offset": 0,
            "first": first,
            "filterFda": False,
            "sortBy": "pvalue_mimic",
            "filterKo": False,
            "topN": 1000,
            "pvalueLe": 0.05,
            "genesUp": genes_up,
            "genesDown": genes_down,
        },
        "query": """query PairEnrichmentQuery(
            $genesUp: [String]!
            $genesDown: [String]!
            $filterTerm: String = \"\"
            $offset: Int = 0
            $first: Int = 10
            $filterFda: Boolean = false
            $sortBy: String = \"\"
            $filterKo: Boolean = false
            $topN: Int = 10000
            $pvalueLe: Float = 0.05
        ) {
            currentBackground {
                pairedEnrich(
                    filterTerm: $filterTerm
                    offset: $offset
                    first: $first
                    filterFda: $filterFda
                    sortby: $sortBy
                    filterKo: $filterKo
                    topN: $topN
                    pvalueLe: $pvalueLe
                    genesDown: $genesDown
                    genesUp: $genesUp
                ) {
                    totalCount
                    consensusCount
                    consensus {
                        drug oddsRatio pvalue adjPvalue approved
                        countSignificant countInsignificant countUpSignificant
                        pvalueUp adjPvalueUp oddsRatioUp
                        pvalueDown adjPvalueDown oddsRatioDown
                    }
                    nodes {
                        pvalueMimic adjPvalueMimic oddsRatioMimic mimickerOverlap
                        pvalueReverse adjPvalueReverse oddsRatioReverse reverserOverlap
                        geneSet {
                            nodes {
                                id nGeneIds term
                                geneSetFdaCountsById {
                                    nodes { count approved }
                                }
                            }
                        }
                    }
                }
            }
        }""",
    }

    response = requests.post(URL, json=query)
    response.raise_for_status()
    res = response.json()

    paired = res["data"]["currentBackground"]["pairedEnrich"]
    df_consensus = pd.DataFrame(paired["consensus"]).rename(
        columns={
            "drug": "perturbation",
            "pvalueUp": "pvalueMimic",
            "pvalueDown": "pvalueReverse",
            "adjPvalueUp": "adjPvalueMimic",
            "adjPvalueDown": "adjPvalueReverse",
            "oddsRatioUp": "oddsRatioMimic",
            "oddsRatioDown": "oddsRatioReverse",
        }
    )
    df_enrichment = pd.DataFrame(paired["nodes"])
    if df_enrichment.empty:
        return pd.DataFrame(), pd.DataFrame()

    df_enrichment["term"] = df_enrichment["geneSet"].map(
        lambda t: t["nodes"][0]["term"].split(" ")[0]
    )
    df_enrichment["geneSetIdUp"] = df_enrichment["geneSet"].map(
        lambda t: next(
            (n["id"] for n in t["nodes"] if " up" in n["term"]), None
        )
    )
    df_enrichment["geneSetIdDown"] = df_enrichment["geneSet"].map(
        lambda t: next(
            (n["id"] for n in t["nodes"] if " down" in n["term"]), None
        )
    )
    df_enrichment = df_enrichment.drop(columns=["geneSet"])
    return df_enrichment, df_consensus

In [ ]:
genes_up = ["STAT1", "STAT3", "JAK1", "JAK2"]
genes_down = ["IFNG", "IRF1", "SOCS1", "SOCS3"]

df_pair, df_pair_consensus = enrich_up_down(genes_up, genes_down)
print(f"Results: {len(df_pair)} enriched signatures")
df_pair.head(10)

## 3. Get Overlapping Genes

Retrieve the overlapping genes between your query and a Perturb-Seqr gene set using its ID (returned from the enrichment queries above).

In [ ]:
def get_overlap(genes: list[str], gene_set_id: str) -> list[str]:
    """Get overlapping genes between a user gene set and a Perturb-Seqr gene set."""
    query = {
        "operationName": "OverlapQuery",
        "variables": {"id": gene_set_id, "genes": genes},
        "query": """query OverlapQuery($id: UUID!, $genes: [String]!) {
            geneSet(id: $id) {
                overlap(genes: $genes) {
                    nodes { symbol ncbiGeneId description summary }
                }
            }
        }""",
    }
    response = requests.post(URL, json=query)
    response.raise_for_status()
    res = response.json()
    return [
        item["symbol"]
        for item in res["data"]["geneSet"]["overlap"]["nodes"]
    ]


def get_up_down_overlap(
    genes_up: list[str],
    genes_down: list[str],
    id_up: str,
    id_down: str,
    overlap_type: str,
) -> list[str]:
    """Get mimicker or reverser overlap for up/down results."""
    if overlap_type == "mimicker":
        up_up = get_overlap(genes_up, id_up)
        dn_dn = get_overlap(genes_down, id_down)
        return list(set(up_up) | set(dn_dn))
    elif overlap_type == "reverser":
        up_dn = get_overlap(genes_up, id_down)
        dn_up = get_overlap(genes_down, id_up)
        return list(set(up_dn) | set(dn_up))
    else:
        raise ValueError("overlap_type must be 'mimicker' or 'reverser'")

In [ ]:
# Use a gene set ID from the single enrichment results
gene_set_id = df_enrichment.iloc[0]["id"]
term = df_enrichment.iloc[0]["term"]
print(f"Getting overlap with: {term}")

overlapping = get_overlap(genes, gene_set_id)
print(f"Overlapping genes: {overlapping}")

## 4. Validate Genes Against Background

Check which genes from your query overlap with the Perturb-Seqr background and retrieve their standardized symbols.

In [ ]:
def get_valid_genes(genes: list[str]) -> list[str]:
    """Find which genes overlap with the Perturb-Seqr background."""
    query = {
        "operationName": "GenesQuery",
        "variables": {"genes": genes},
        "query": """query GenesQuery($genes: [String]!) {
            geneMap2(genes: $genes) {
                nodes {
                    gene
                    geneInfo { symbol }
                }
            }
        }""",
    }
    response = requests.post(URL, json=query)
    response.raise_for_status()
    res = response.json()
    return [
        g["geneInfo"]["symbol"]
        for g in res["data"]["geneMap2"]["nodes"]
        if g["geneInfo"] is not None
    ]

In [ ]:
valid = get_valid_genes(genes)
print(f"Valid genes: {valid}")